# VietLens - Continual Fine-Tuning Qwen2.5-VL-3B LoRA
Notebook này thực hiện **Fine-tune nâng cao (Continual Learning)** cho mô hình thị giác `Qwen/Qwen2.5-VL-3B-Instruct` trên tập dữ liệu ảnh di sản mới được làm sạch và đóng gói (`packaged_dataset.zip`).

### Điểm nổi bật:
- **Continual Learning:** Kế thừa bộ trọng số nền tảng `imbee510/qwen2-5-vl-landmark-vietnam-lora` để giữ vững tri thức toàn quốc và tối ưu hóa độ nhạy cho 14 địa danh trọng điểm Đà Nẵng & Miền Trung.
- **Tối ưu VRAM:** Sử dụng **4-bit QLoRA (BitsAndBytes)** kết hợp `paged_adamw_8bit`, chạy mượt mà trên **Google Colab T4 GPU (15GB)** với VRAM tiêu thụ < 9 GB.
- **Tương thích 100%:** Sau khi huấn luyện, bộ trọng số mới có thể đưa ngay vào `colab_vision_server.ipynb` phục vụ Mobile App VietLens.


In [1]:
# 1. CÀI ĐẶT THƯ VIỆN CẦN THIẾT (TỐI ƯU CHO QWEN2.5-VL & T4 GPU)
!pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils trl datasets
!pip install -q -U pillow torchvision

import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('⚠️ Chưa bật GPU! Hãy chọn Runtime -> Change runtime type -> T4 GPU.')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
ERROR: Operation cancelled by user
CUDA Available: True
GPU Device: Tesla T4
GPU VRAM: 14.56 GB


## 2. Giải nén Tập Dữ Liệu (`packaged_dataset.zip`)
Bạn hãy đảm bảo file `packaged_dataset.zip` đã được upload vào thư mục hiện tại của Colab (hoặc mount từ Google Drive).


In [4]:
import os
import zipfile
from pathlib import Path

# Thư mục giải nén đích trên máy ảo Colab
DATASET_DIR = Path('./dataset')

# Danh sách các vị trí tự động nhận diện file zip
CANDIDATE_PATHS = [
    Path('packaged_dataset.zip'),
    Path('/content/packaged_dataset.zip'),
    Path('/content/drive/MyDrive/packaged_dataset.zip'),
    Path('/content/drive/MyDrive/VietLens/packaged_dataset.zip'),
    Path('/content/drive/MyDrive/Colab Notebooks/packaged_dataset.zip'),
]

zip_path = None
for p in CANDIDATE_PATHS:
    if p.exists():
        zip_path = p
        break

# Nếu chưa có sẵn, tự động kết nối Google Drive để tìm kiếm
if zip_path is None:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print('⏳ Đang kết nối tới Google Drive...')
            drive.mount('/content/drive')
        
        drive_root = Path('/content/drive/MyDrive')
        if drive_root.exists():
            for p in CANDIDATE_PATHS:
                if p.exists():
                    zip_path = p
                    break
            if zip_path is None:
                found_files = list(drive_root.glob('**/packaged_dataset.zip'))
                if found_files:
                    zip_path = found_files[0]
    except Exception as e:
        print(f'ℹ️ Google Drive info: {e}')

# Tiến hành giải nén khi đã tìm thấy file zip
if zip_path and zip_path.exists():
    print(f'📦 Đã tìm thấy dataset tại: {zip_path.resolve()}')
    print('⏳ Đang giải nén vào ./dataset ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print(f'✅ Giải nén thành công vào thư mục {DATASET_DIR.resolve()}')
    
    # Thống kê nhanh dữ liệu
    train_f = DATASET_DIR / 'danang_train.jsonl'
    val_f = DATASET_DIR / 'danang_val.jsonl'
    test_f = DATASET_DIR / 'danang_test.jsonl'
    
    def count_lines(p):
        return sum(1 for line in open(p, 'r', encoding='utf-8') if line.strip()) if p.exists() else 0
    
    print(f' - Tập Train: {count_lines(train_f)} mẫu')
    print(f' - Tập Val:   {count_lines(val_f)} mẫu')
    print(f' - Tập Test:  {count_lines(test_f)} mẫu')
else:
    print('❌ Vẫn chưa tìm thấy file packaged_dataset.zip trên Colab!')
    print('👉 Hãy làm theo 1 trong 2 cách cực kỳ đơn giản sau:')
    print('   1. [Khuyên dùng]: Mở drive.google.com trên trình duyệt -> Kéo file packaged_dataset.zip vào Google Drive của bạn. Sau đó bấm chạy lại cell này, code sẽ tự động nhận diện!')
    print('   2. [Trong VS Code]: Chuột phải vào file packaged_dataset.zip ở Explorer bên trái -> Chọn "Upload to Colab Session".')


📦 Đã tìm thấy dataset tại: /content/drive/MyDrive/packaged_dataset.zip
⏳ Đang giải nén vào ./dataset ...
✅ Giải nén thành công vào thư mục /content/dataset
 - Tập Train: 380 mẫu
 - Tập Val:   44 mẫu
 - Tập Test:  58 mẫu


## 🤖 3. Tải Base Model & Trọng số LoRA Ban Đầu (Continual Learning)
Chúng ta tải `Qwen/Qwen2.5-VL-3B-Instruct` dạng 4-bit, sau đó nạp adapter LoRA `imbee510/qwen2-5-vl-landmark-vietnam-lora` với cờ `is_trainable=True` để tiếp tục cập nhật trọng số.


In [1]:
import os
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from peft import PeftModel, prepare_model_for_kbit_training

# Tránh phân mảnh bộ nhớ VRAM
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

BASE_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
INITIAL_LORA_ID = 'imbee510/qwen2-5-vl-landmark-vietnam-lora'

# Cấu hình 4-bit NF4 giúp tiết kiệm tối đa VRAM trên T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'⏳ 1/3 Đang tải Base Model: {BASE_MODEL_ID}...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto'
)

print(f'⏳ 2/3 Đang tải Processor (khống chế token thị giác tối đa 512x512)...')
# Giới hạn kích thước ảnh để không bị bùng nổ token kích hoạt gây OOM
min_pixels = 256 * 28 * 28
max_pixels = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    min_pixels=min_pixels,
    max_pixels=max_pixels
)

# Chuẩn bị mô hình cho huấn luyện kbit kết hợp gradient checkpointing
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

print(f'⏳ 3/3 Đang tải Adapter LoRA: {INITIAL_LORA_ID} (is_trainable=True)...')
model = PeftModel.from_pretrained(
    base_model,
    INITIAL_LORA_ID,
    is_trainable=True
)

# Tắt use_cache khi huấn luyện với gradient checkpointing
model.config.use_cache = False

model.print_trainable_parameters()
print('✅ Đã sẵn sàng Continual LoRA Fine-Tuning!')


⏳ 1/3 Đang tải Base Model: Qwen/Qwen2.5-VL-3B-Instruct...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

⏳ 2/3 Đang tải Processor (khống chế token thị giác tối đa 512x512)...


⏳ 3/3 Đang tải Adapter LoRA: imbee510/qwen2-5-vl-landmark-vietnam-lora (is_trainable=True)...
trainable params: 37,431,296 || all params: 3,792,054,272 || trainable%: 0.9871
✅ Đã sẵn sàng Continual LoRA Fine-Tuning!


## 📊 4. Xây Dựng Dataset & Data Collator Đa Phương Thức
Dùng `qwen-vl-utils.process_vision_info` để nạp trực tiếp ảnh PIL và tokenize hội thoại chuẩn xác.


In [5]:
import json
from PIL import Image
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class QwenVLDataset(Dataset):
    def __init__(self, jsonl_path, image_base_dir):
        self.samples = []
        self.image_base_dir = Path(image_base_dir)
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    self.samples.append(json.loads(line.strip()))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        rel_img_path = item['image']
        full_path = self.image_base_dir / rel_img_path
        
        try:
            img = Image.open(full_path).convert('RGB')
        except Exception as e:
            print(f'Lỗi mở ảnh {full_path}: {e}')
            img = Image.new('RGB', (224, 224), (0, 0, 0))

        messages = []
        for msg in item['messages']:
            role = msg['role']
            if role == 'user':
                user_content = []
                for c in msg['content']:
                    if c.get('type') == 'image':
                        user_content.append({'type': 'image', 'image': img})
                    else:
                        user_content.append(c)
                messages.append({'role': role, 'content': user_content})
            else:
                messages.append(msg)
        return messages

# Khởi tạo Dataset Train và Val
train_img_dir = DATASET_DIR / 'landmark_vietnam_dataset_split' / 'train'
val_img_dir = DATASET_DIR / 'landmark_vietnam_dataset_split' / 'val'

train_dataset = QwenVLDataset(DATASET_DIR / 'danang_train.jsonl', train_img_dir)
val_dataset = QwenVLDataset(DATASET_DIR / 'danang_val.jsonl', val_img_dir)
print(f'✅ Đã load {len(train_dataset)} mẫu train, {len(val_dataset)} mẫu val.')

def custom_data_collator(batch):
    texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in batch]
    image_inputs, video_inputs = process_vision_info(batch)
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt'
    )
    labels = inputs['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    if image_token_id is not None:
        labels[labels == image_token_id] = -100
    inputs['labels'] = labels
    return inputs


✅ Đã load 380 mẫu train, 44 mẫu val.


## 🚀 5. Huấn Luyện Mô Hình (Training)
Cấu hình Learning Rate `1.5e-4` (thấp và an toàn cho Continual Learning), `per_device_train_batch_size=2`, `gradient_accumulation_steps=4`, tối ưu hóa bằng `paged_adamw_8bit`.


In [6]:
import gc
import torch
from transformers import TrainingArguments, Trainer

# Dọn sạch VRAM cache tồn đọng từ các lần chạy trước
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = './qwen2_5_vl_danang_lora'

# Cấu hình huấn luyện tối ưu chống OOM trên T4 GPU (15GB)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,            # Giảm về 1 trên T4
    gradient_accumulation_steps=8,            # Tăng lên 8 để giữ nguyên effective batch size = 8
    learning_rate=1.5e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,              # Cực kỳ quan trọng: Giảm 70% VRAM
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    dataloader_num_workers=0,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_data_collator
)

print('🚀 Bắt đầu quá trình huấn luyện Continual Learning...')
trainer.train()
print('🎉 Huấn luyện thành công!')


🚀 Bắt đầu quá trình huấn luyện Continual Learning...


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.304468,0.309315
2,0.298114,0.298860
3,0.297399,0.298112


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': 

🎉 Huấn luyện thành công!


## 💾 6. Lưu Adapter Trọng Số Mới
Lưu adapter LoRA vào thư mục cục bộ và đóng gói thành file `.zip` để dễ dàng tải về máy tính.


In [7]:
import shutil

print('⏳ Đang lưu adapter LoRA và Processor...')
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

zip_output_name = 'qwen2_5_vl_danang_lora.zip'
shutil.make_archive('qwen2_5_vl_danang_lora', 'zip', OUTPUT_DIR)
print(f'✅ Đã nén thành công: {zip_output_name} ({os.path.getsize(zip_output_name)/(1024*1024):.2f} MB)')


⏳ Đang lưu adapter LoRA và Processor...
✅ Đã nén thành công: qwen2_5_vl_danang_lora.zip (330.91 MB)


In [8]:
import shutil
import os

# Đường dẫn trên Google Drive
drive_dest = '/content/drive/MyDrive/qwen2_5_vl_danang_lora.zip'

if os.path.exists('/content/drive/MyDrive'):
    print("⏳ Đang sao chép file sang Google Drive...")
    shutil.copy2('qwen2_5_vl_danang_lora.zip', drive_dest)
    print(f"✅ ĐÃ LƯU THÀNH CÔNG VÀO GOOGLE DRIVE TẠI: {drive_dest}")
    print("👉 Bạn chỉ cần mở drive.google.com trên trình duyệt là thấy file và bấm Tải về thoải mái!")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy2('qwen2_5_vl_danang_lora.zip', drive_dest)
    print("✅ Đã kết nối và lưu thành công vào Google Drive!")

⏳ Đang sao chép file sang Google Drive...
✅ ĐÃ LƯU THÀNH CÔNG VÀO GOOGLE DRIVE TẠI: /content/drive/MyDrive/qwen2_5_vl_danang_lora.zip
👉 Bạn chỉ cần mở drive.google.com trên trình duyệt là thấy file và bấm Tải về thoải mái!


## 📈 7. Đánh Giá Độ Chính Xác Trên Tập Test (58 Ảnh)
Chạy kiểm thử trên tập `danang_test.jsonl` độc lập (mô hình chưa từng nhìn thấy trong lúc train) để đo lường độ chính xác Top-1.


In [9]:
from tqdm import tqdm

test_f = DATASET_DIR / 'danang_test.jsonl'
test_img_dir = DATASET_DIR / 'landmark_vietnam_dataset_split' / 'test'

test_samples = []
with open(test_f, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            test_samples.append(json.loads(line.strip()))

print(f'📊 Đang đánh giá trên {len(test_samples)} mẫu Test...')
model.eval()

correct_count = 0
results = []

for s in tqdm(test_samples):
    img_path = test_img_dir / s['image']
    if not img_path.exists():
        continue
        
    img = Image.open(img_path).convert('RGB')
    expected_answer = s['messages'][2]['content'][0]['text']
    
    # Tạo prompt suy luận
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': 'You are a highly capable AI assistant specialized in identifying global landmarks.'}]},
        {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': 'What is the name of the place in this photo?'}]}
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to('cuda')
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    
    # Kiểm tra so khớp tên địa danh
    # Chuẩn hóa so sánh không phân biệt hoa thường
    is_correct = False
    # Lấy tên địa danh kỳ vọng (giữa 'this is ' và ',')
    exp_clean = expected_answer.replace('Based on the visual features, this is ', '').strip()
    exp_landmark = exp_clean.split(',')[0].strip().lower()
    
    if exp_landmark in output_text.lower():
        is_correct = True
        correct_count += 1
        
    results.append({
        'image': s['image'],
        'expected': expected_answer,
        'predicted': output_text,
        'is_correct': is_correct
    })

acc = (correct_count / len(test_samples)) * 100 if test_samples else 0
print('=' * 60)
print(f'🎯 KẾT QUẢ ĐÁNH GIÁ TẬP TEST: {correct_count}/{len(test_samples)} ({acc:.2f}%)')
print('=' * 60)

# In 10 mẫu dự đoán đầu tiên để kiểm tra
for r in results[:10]:
    icon = '✅' if r['is_correct'] else '❌'
    print(f"{icon} [{r['image']}]")
    print(f"   Expected : {r['expected']}")
    print(f"   Predicted: {r['predicted']}")


📊 Đang đánh giá trên 58 mẫu Test...


  0%|          | 0/58 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
 16%|█▌        | 9/58 [00:33<03:00,  3.69s/it]/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
100%|██████████| 58/58 [03:40<00:00,  3.80s/it]

🎯 KẾT QUẢ ĐÁNH GIÁ TẬP TEST: 53/58 (91.38%)
✅ [asia_park/test_asia_park_1.jpg]
   Expected : Based on the visual features, this is Sun World Asia Park, Da Nang.
   Predicted: Based on the visual features, this is Sun World Asia Park, Da Nang.
✅ [asia_park/test_asia_park_2.jpg]
   Expected : Based on the visual features, this is Sun World Asia Park, Da Nang.
   Predicted: Based on the visual features, this is Sun World Asia Park, Da Nang.
✅ [asia_park/test_asia_park_3.jpg]
   Expected : Based on the visual features, this is Sun World Asia Park, Da Nang.
   Predicted: Based on the visual features, this is Sun World Asia Park, Da Nang.
✅ [asia_park/test_asia_park_4.jpg]
   Expected : Based on the visual features, this is Sun World Asia Park, Da Nang.
   Predicted: Based on the visual features, this is Sun World Asia Park, Da Nang.
❌ [asia_park/test_asia_park_5.jpg]
   Expected : Based on the visual features, this is Sun World Asia Park, Da Nang.
   Predicted: Based on the visual features,

## 🎯 8. Hướng Dẫn Tích Hợp Vào `colab_vision_server.ipynb`
Sau khi hoàn tất huấn luyện và lưu adapter:
1. Tải file `qwen2_5_vl_danang_lora.zip` về máy tính.
2. Trong `VLensModel/colab_vision_server.ipynb`, đổi đường dẫn:
   ```python
   LORA_PATH = './qwen2_5_vl_danang_lora'  # Trỏ trực tiếp đến thư mục adapter mới giải nén
   ```
3. Chạy Server Vision qua Ngrok -> Ứng dụng VietLens sẽ lập tức nhận diện chuẩn xác các địa danh mới với độ tự tin vượt trội!
